<a href="https://colab.research.google.com/github/huwperkins08-lang/TSF_seepage_detection_ElSoldado/blob/main/REP_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from geemap import Map
import ee
import geemap.core as geemap
ee.Authenticate()
ee.Initialize(project='sxe390-el-soldado')

#define dataset and asign satellite
El_soldado_trial = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

#Define area of interest
mine_site = ee.Geometry.Point([-71.16,-32.64])

#define and filter data
El_soldado_23_24 = (El_soldado_trial
                    .filterBounds(mine_site)
                    .filterDate('2023-12-01','2024-04-01')
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',40)))

#count number of returned images
count = El_soldado_23_24.size()
print('Number of images found:', count.getInfo())

# create median composite and rescale to 0-1 reflectance
El_soldado_23_24_median = El_soldado_23_24.median().multiply(0.0001)

#define study area
study_area = mine_site.buffer(20000)

#clip median image to study area
El_soldado_clipped = El_soldado_23_24_median.clip(study_area)

#display median image using geemap
#1 define map
map = geemap.Map(center=[-32.64,-71.16], zoom=13)

#add image to map
viz_params = {'bands':['B4','B3','B2'], 'min':0, 'max':0.3}
map.addLayer(El_soldado_clipped, viz_params,'El SOldado Median 23-24')

#Calculate NDVI using B8 and B4
ndvi = El_soldado_clipped.normalizedDifference(['B8','B4']).rename('NDVI')

#create NDVI mask
NDVI_mask = ndvi.gt(0.3)

#calculate REP
#formula - 700 + 40 *((((B4 + B7)/2) - B5)/(B6 - B5))
rep = El_soldado_23_24_median.expression(
    '700 + 40 * (((B4 + B7)/2) - B5)/(B6 - B5)',{
        'B4': El_soldado_23_24_median.select('B4'),
        'B5': El_soldado_23_24_median.select('B5'),
        'B6': El_soldado_23_24_median.select('B6'),
        'B7': El_soldado_23_24_median.select('B7')
    }).rename('REP')

#apply NDVI mask to REP data
rep_masked = rep.updateMask(NDVI_mask)

#display REP on map
# healthy vegetation REP = 715-725, stressed vegetation drops towartd 705
rep_params = {
    'min':705,
    'max':725,
    'palette':['red','orange','yellow','green']
}
map.addLayer(rep_masked, rep_params, 'REP')

#calculate vegetation

#apply mask to layer
ndvi_masked_vegetation = ndvi.updateMask(NDVI_mask)

#applu lauyer to map
map.addLayer(ndvi_masked_vegetation, {'min':0.3, 'max':1, 'palette':['yellow', 'green']}, 'Vegetation Only')



map

Number of images found: 12


Map(center=[-32.64, -71.16], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

In [ ]:
import ee
import geemap.core as geemap

# 1. INITIALIZATION
ee.Authenticate()
ee.Initialize(project='sxe390-el-soldado')

# 2. DEFINE LOCATION & STUDY AREA (The "Ethical Clip")
# El Soldado Mine, Chile
lat, lon = -32.65, -71.16
mine_site = ee.Geometry.Point([lon, lat])
study_area = mine_site.buffer(20000) # 5km buffer to limit computing power

# 3. CLOUD MITIGATION FUNCTION (QA60)
def mask_s2_clouds(image):
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    # Return masked image and scale to 0-1 reflectance
    return image.updateMask(mask).divide(10000)

# 4. SOURCE & FILTER DATA
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(mine_site)
                  .filterDate('2023-12-01', '2024-04-01')
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
                  .map(mask_s2_clouds))

# Create the Median (Noise Reduction) and Clip
base_median = s2_collection.median().clip(study_area)

# 5. TOPOGRAPHIC & SPECTRAL NORMALIZATION
# We use a Band Ratio approach to level out terrain shading
# Normalizing by the mean of all bands reduces the 'hill-shadow' effect
img_mean = base_median.reduce(ee.Reducer.mean())
topo_corrected = base_median.divide(img_mean)

# 6. NDVI MASKING (Vegetation Isolation)
ndvi = base_median.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndvi_mask = ndvi.gt(0.3)

# 7. CONTINUUM REMOVAL (CR) & REP CALCULATION
# Isolate chlorophyll from background interference
continuum = base_median.select('B7')
b4_cr = base_median.select('B4').divide(continuum)
b5_cr = base_median.select('B5').divide(continuum)
b6_cr = base_median.select('B6').divide(continuum)
b7_cr = ee.Image(1) # B7/B7 reference point

# Calculate REP on the CR-cleaned bands
rep_cr = ee.Image(700).add(
    ee.Image(40).multiply(
        ( (b4_cr.add(b7_cr).divide(2)).subtract(b5_cr) ).divide(b6_cr.subtract(b5_cr))
    )
).rename('REP_CR').updateMask(ndvi_mask)

# 8. VISUALIZATION
Map = geemap.Map(center=[-32.65, -71.16], zoom=13)

# Layer 1: Natural Color for context
Map.addLayer(base_median, {'bands':['B4','B3','B2'], 'min':0, 'max':0.3}, '1. Natural Color')

# Layer 2: The Final Product (REP with CR and NDVI Mask)
rep_params = {'min': 705, 'max': 725, 'palette': ['red', 'orange', 'yellow', 'green']}
Map.addLayer(rep_cr, rep_params, '2. REP (Continuum Removed)')

# 1. Define two comparison points
# Impact site (near the tailings/mine)
impact_pt = ee.Geometry.Point([-71.18, -32.64]).buffer(300)

# Control site (away from the mine in a similar valley)
control_pt = ee.Geometry.Point([-71.27, -32.70]).buffer(300)

# 2. Extract the Mean REP for both
impact_stats = rep_cr.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=impact_pt,
    scale=10
)

control_stats = rep_cr.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=control_pt,
    scale=10
)

# 3. Print the results to the console
print('--- RESULTS ---')
print('Impact Site Mean REP:', impact_stats.get('REP_CR').getInfo())
print('Control Site Mean REP:', control_stats.get('REP_CR').getInfo())

# 4. Add the circles to the map so you can see where they are
Map.addLayer(impact_pt, {'color': 'red'}, 'Impact Buffer (300m)', False)
Map.addLayer(control_pt, {'color': 'blue'}, 'Control Buffer (300m)', False)

Map

--- RESULTS ---
Impact Site Mean REP: 716.714212528223
Control Site Mean REP: 716.9238114149348


Map(center=[-32.65, -71.16], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…